# XE4 grouped GEMM (tensor-desc update)

A fully worked, cell-by-cell functional port of the sycl-tla example [`examples/cute/tutorial/xe4/grouped_gemm_adma_tensor_desc_update.cpp`](../../examples/cute/tutorial/xe4/grouped_gemm_adma_tensor_desc_update.cpp ), runnable on the CPU.

Several independent GEMMs in one launch; per-group ADMA descriptors.

Every cell performs one operation on small concrete data and shows the matching Xe layout. Run them top to bottom. (A runnable script version lives in `grouped_gemm_adma_tensor_desc_update.py`.)

## Setup — data + a tiny layout printer

In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True, linewidth=120)
from tensor_layouts import Layout, size
from tensor_layouts.analysis import is_bijective
from tensor_layouts.atoms_xe_common import make_slm_layout_elem, sizeof_bits

def show_layout(layout, n_rows, n_cols, rl="m", cl="k", max_r=8, max_c=8):
    "Print coord -> memory offset for a rank-2 layout (truncated)."
    R, C = min(n_rows, max_r), min(n_cols, max_c)
    print("      " + "".join((cl + str(j)).ljust(5) for j in range(C)) + (" ..." if C < n_cols else ""))
    for i in range(R):
        print((" " + rl + str(i)).ljust(6) + "".join(str(layout(i, j)).ljust(5) for j in range(C))
              + (" ..." if C < n_cols else ""))
    if R < n_rows:
        print("  ...  (%dx%d total)" % (n_rows, n_cols))

M, N, K = 32, 32, 16          # A rows, B rows, contraction
rng = np.random.default_rng(0)
A = rng.integers(-2, 3, size=(M, K)).astype(np.float32)   # A  (M x K)
B = rng.integers(-2, 3, size=(N, K)).astype(np.float32)   # B  (N x K), used as B^T
print("A", A.shape, " B", B.shape)
print("A[:4]:\n", A[:4])

A (32, 16)  B (32, 16)
A[:4]:
 [[ 2.  1.  0. -1. -1. -2. -2. -2. -2.  2.  1.  2.  0.  1.  2.  1.]
 [ 1.  0.  0.  2. -1.  2.  1. -2. -1.  2.  0. -2.  1.  1.  2. -2.]
 [-2.  2. -2.  0. -2. -1.  0.  0.  0. -2. -2. -2. -2.  1.  0.  1.]
 [-1.  1.  1. -1.  0.  2.  2.  2. -1.  1.  2.  1.  2.  1.  1. -1.]]


## Step — grouped GEMM

In [2]:
# Grouped GEMM: several independent problems share one launch. Each group has
# its own A/B (different sizes) and its own ADMA tensor descriptor.
groups = [(24, 16, 8), (32, 8, 16)]          # (M, N, K) per group
for gi, (m, n, k) in enumerate(groups):
    Ag = rng.integers(-2, 3, size=(m, k)).astype(np.float32)
    Bg = rng.integers(-2, 3, size=(n, k)).astype(np.float32)
    Sg = Ag @ Bg.T
    print("group", gi, "M=%d N=%d K=%d ->" % (m, n, k), "S", Sg.shape, " S[0,:4]", Sg[0, :4])

group 0 M=24 N=16 K=8 -> S (24, 16)  S[0,:4] [-8. -1.  7.  4.]
group 1 M=32 N=8 K=16 -> S (32, 8)  S[0,:4] [-5. -4. 14.  2.]


## Recap

loop over groups, one GEMM each.